# HRV Longitudinal Mapping

Analyse long-term Heart Rate Variability trends from the VEYN SQLite store.

## Setup
```bash
pip install pandas matplotlib seaborn scipy
```

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

DB_PATH = '../veyn.db'
conn = sqlite3.connect(DB_PATH)
print('Connected to', DB_PATH)

In [ ]:
# Load baseline samples for HRV
query = """
SELECT device_id, metric, ts_ms, value
FROM baseline_samples
WHERE metric IN ('hrv', 'heart_rate')
ORDER BY ts_ms
"""
df = pd.read_sql_query(query, conn)
df['datetime'] = pd.to_datetime(df['ts_ms'], unit='ms')
df['date'] = df['datetime'].dt.date
print(f'Loaded {len(df)} samples')
df.head()

In [ ]:
# Daily HRV statistics
hrv = df[df['metric'] == 'hrv'].copy()
daily = hrv.groupby('date')['value'].agg(['mean', 'std', 'count'])
daily.columns = ['hrv_mean', 'hrv_std', 'sample_count']

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(daily.index, daily['hrv_mean'], linewidth=2, label='Daily mean HRV')
ax.fill_between(
    daily.index,
    daily['hrv_mean'] - daily['hrv_std'],
    daily['hrv_mean'] + daily['hrv_std'],
    alpha=0.2, label='±1 std'
)
ax.set_xlabel('Date')
ax.set_ylabel('HRV (ms)')
ax.set_title('HRV Longitudinal Trend')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 7-day vs 30-day rolling averages (baseline drift detection)
hrv_ts = hrv.set_index('datetime')['value'].sort_index()
rolling_7d = hrv_ts.rolling('7D').mean()
rolling_30d = hrv_ts.rolling('30D').mean()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(rolling_7d.index, rolling_7d, label='7-day rolling', linewidth=2)
ax.plot(rolling_30d.index, rolling_30d, label='30-day rolling', linewidth=2, alpha=0.7)
ax.set_ylabel('HRV (ms)')
ax.set_title('Baseline Drift: 7-day vs 30-day HRV')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# HRV distribution by intent_code (from memory records)
mem_query = """
SELECT intent_at_time, hrv_at_time, hr_at_time, confidence_at_time
FROM memory_records
WHERE hrv_at_time IS NOT NULL
"""
mem = pd.read_sql_query(mem_query, conn)

fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=mem, x='intent_at_time', y='hrv_at_time', ax=ax)
ax.set_title('HRV Distribution by Intent Code')
ax.set_xlabel('Intent Code')
ax.set_ylabel('HRV (ms)')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
conn.close()
print('Done.')